# 03 — Setup Ollama + Qwen3.5

Verifica a infraestrutura de inferência local antes dos experimentos.

- Modelo: **qwen3.5:9b** (6.6 GB VRAM)
- Backend: **Ollama** em `http://localhost:11434`
- GPU: NVIDIA RTX 5070 (12 GB VRAM)

## 1. Imports

In [1]:
import json
import subprocess
import time

import polars as pl
import requests

OLLAMA_BASE = "http://127.0.0.1:11434"
MODEL = "qwen3.5:9b"

## 2. Verificar conexão com Ollama

In [2]:
try:
    r = requests.get(OLLAMA_BASE, timeout=5)
    print(f"Status: {r.status_code}")
    print(f"Resposta: {r.text}")
except requests.exceptions.ConnectionError:
    print("ERRO: Ollama não está rodando.")
    print("Inicie com: ollama serve")

Status: 200
Resposta: Ollama is running


## 3. Listar modelos disponíveis

In [3]:
r = requests.get(f"{OLLAMA_BASE}/api/tags")
models = r.json().get("models", [])

print(f"{'Modelo':<30} {'Tamanho':>12}")
print("-" * 44)
for m in models:
    size_gb = m["size"] / 1e9
    print(f"{m['name']:<30} {size_gb:>10.1f} GB")

Modelo                              Tamanho
--------------------------------------------
qwen3.5:9b                            6.6 GB


## 4. Verificar se qwen3.5:9b está disponível

In [4]:
names = [m["name"] for m in models]

if MODEL in names:
    print(f"OK  {MODEL} disponível")
else:
    print(f"AVISO: {MODEL} não encontrado.")
    print(f"Execute no terminal: ollama pull {MODEL}")

OK  qwen3.5:9b disponível


## 5. Verificar GPU

In [5]:
result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.free,utilization.gpu",
     "--format=csv,noheader"],
    capture_output=True, text=True
)
if result.returncode == 0:
    name, total, free, util = result.stdout.strip().split(", ")
    print(f"GPU:         {name}")
    print(f"VRAM total:  {total}")
    print(f"VRAM livre:  {free}")
    print(f"Utilização:  {util}")
else:
    print("AVISO: nvidia-smi não encontrado ou sem GPU NVIDIA.")

GPU:         NVIDIA GeForce RTX 5070
VRAM total:  12227 MiB
VRAM livre:  2775 MiB
Utilização:  1 %


## 6. Teste de inferência e benchmark

In [6]:
TWEET_TESTE = "Você é um idiota e deveria se calar."

payload = {
    "model": MODEL,
    "prompt": (
        "Você é um sistema de moderação de conteúdo em português brasileiro.\n"
        "Classifique o comentário abaixo em UMA das categorias:\n"
        "NOT_TOXIC, OBSCENE, INSULT, HOMOPHOBIA, RACISM, MISOGYNY, XENOPHOBIA\n"
        "Responda APENAS com o nome da categoria.\n\n"
        f"Comentário: {TWEET_TESTE}\n"
        "Classificação:"
    ),
    "stream": False,
    "think": False,
}

print("Enviando prompt...")
t0 = time.time()
r = requests.post(f"{OLLAMA_BASE}/api/generate", json=payload)
elapsed = time.time() - t0

data = r.json()
tps = data["eval_count"] / (data["eval_duration"] / 1e9)

print(f"\nTweet:          {TWEET_TESTE}")
print(f"Classificação:  {data['response'].strip()}")
print(f"Tokens gerados: {data['eval_count']}")
print(f"Tokens/s:       {tps:.1f}")
print(f"Latência total: {elapsed:.2f}s")

Enviando prompt...



Tweet:          Você é um idiota e deveria se calar.
Classificação:  NOT_TOXIC
Tokens gerados: 5
Tokens/s:       98.1
Latência total: 0.41s


## 7. Teste com 5 tweets da amostra

In [7]:
sample = pl.read_csv("../data/sample/toldBr_sample_500.csv").sample(n=5, seed=99)

resultados = []
for row in sample.iter_rows(named=True):
    tweet = row["text"]
    label_real = row["label"]

    payload = {
        "model": MODEL,
        "prompt": (
            "Você é um sistema de moderação de conteúdo em português brasileiro.\n"
            "Classifique o comentário abaixo em UMA das categorias:\n"
            "NOT_TOXIC, OBSCENE, INSULT, HOMOPHOBIA, RACISM, MISOGYNY, XENOPHOBIA\n"
            "Responda APENAS com o nome da categoria.\n\n"
            f"Comentário: {tweet}\n"
            "Classificação:"
        ),
        "stream": False,
        "think": False,
    }

    t0 = time.time()
    r = requests.post(f"{OLLAMA_BASE}/api/generate", json=payload)
    elapsed = time.time() - t0

    data = r.json()
    predicao = data["response"].strip()
    tps = data["eval_count"] / (data["eval_duration"] / 1e9)

    resultados.append({
        "tweet": tweet[:60] + "..." if len(tweet) > 60 else tweet,
        "label_real": label_real,
        "predicao": predicao,
        "tokens_s": round(tps, 1),
        "latencia_s": round(elapsed, 2)
    })

df_res = pl.DataFrame(resultados)
print(df_res)

shape: (5, 5)
┌─────────────────────────────────┬────────────┬────────────┬──────────┬────────────┐
│ tweet                           ┆ label_real ┆ predicao   ┆ tokens_s ┆ latencia_s │
│ ---                             ┆ ---        ┆ ---        ┆ ---      ┆ ---        │
│ str                             ┆ str        ┆ str        ┆ f64      ┆ f64        │
╞═════════════════════════════════╪════════════╪════════════╪══════════╪════════════╡
│ ontem parecia que eu estava c … ┆ not_toxic  ┆ NOT_TOXIC  ┆ 102.5    ┆ 0.24       │
│ @user nenhum mal te suceder谩   ┆ not_toxic  ┆ NOT_TOXIC  ┆ 103.0    ┆ 0.24       │
│ n…                              ┆            ┆            ┆          ┆            │
│ cara vai se foder eu vou quebr… ┆ not_toxic  ┆ NOT_TOXIC  ┆ 104.5    ┆ 0.24       │
│ @user @user tropinha é ridicul… ┆ insult     ┆ NOT_TOXIC  ┆ 105.3    ┆ 0.24       │
│ e os gays comemoram             ┆ not_toxic  ┆ HOMOPHOBIA ┆ 97.4     ┆ 0.26       │
└─────────────────────────────────┴──────

## 8. Resumo do setup

In [8]:
tps_medio = df_res["tokens_s"].mean()
lat_media = df_res["latencia_s"].mean()

print("=" * 45)
print("RESUMO DO SETUP")
print("=" * 45)
print(f"Modelo:          {MODEL}")
print(f"VRAM estimada:   ~6.6 GB")
print(f"Tokens/s médio:  {tps_medio:.1f}")
print(f"Latência média:  {lat_media:.2f}s")
print("=" * 45)
print("Setup OK — pronto para os experimentos.")

RESUMO DO SETUP
Modelo:          qwen3.5:9b
VRAM estimada:   ~6.6 GB
Tokens/s médio:  102.5
Latência média:  0.24s
Setup OK — pronto para os experimentos.
